# 📊 Japan Grid Analysis: Data Loading & Inspection

**Notebook 01:** Load synthetic Japan renewable energy grid data and perform basic quality checks.

This notebook:
1. Loads 1 year of hourly renewable generation and demand data
2. Inspects data quality (shape, dtypes, nulls)
3. Shows descriptive statistics and monthly patterns
4. Exports cleaned CSV for downstream analysis

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from data_loader import generate_synthetic_japan_grid

print("✅ Libraries imported successfully")

## Step 1: Load Data

In [ ]:
# Generate 1 year of synthetic data for 10 Japanese regions
df = generate_synthetic_japan_grid(days=365, seed=42)

print(f"✅ Data generated successfully")
print(f"\nDataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Regions covered: {sorted(df['region'].unique())}")
print(f"Date range: {df['datetime'].min().date()} to {df['datetime'].max().date()}")

## Step 2: Data Quality Inspection

In [ ]:
# Display dtypes and null counts
print("Data Types:")
print(df.dtypes)
print("\n" + "="*50)
print("Null Value Counts:")
print(df.isnull().sum())
print(f"\n✅ Total null values: {df.isnull().sum().sum()} (0 = clean data)")

## Step 3: Descriptive Statistics

In [ ]:
# Overall statistics
print("📊 Overall Dataset Statistics:")
print(df.describe())

print("\n" + "="*80)
print("\n📍 Statistics by Region:")
regional_stats = df.groupby('region')[['solar_mw', 'wind_mw', 'demand_mw', 'surplus_mw', 'price_jpy_kwh']].agg(['mean', 'min', 'max', 'std'])
print(regional_stats.round(2))

## Step 4: Visualize Sample Data

In [ ]:
# Plot total Japan solar generation over first 30 days
daily_solar = df.groupby(df['datetime'].dt.date)['solar_mw'].sum().reset_index()
daily_solar.columns = ['date', 'total_solar_mw']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily_solar['date'],
    y=daily_solar['total_solar_mw'],
    mode='lines+markers',
    line=dict(color='gold', width=2),
    name='Total Solar Generation',
    fill='tozeroy'
))

fig.update_layout(
    title="Total Japan Solar Generation (First 30 Days)",
    xaxis_title="Date",
    yaxis_title="Solar Generation (MW)",
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig.show()
print("✅ Chart rendered successfully")

## Step 5: Export Cleaned Data

In [ ]:
# Export to processed data directory
output_path = '../data/processed/japan_grid_clean.csv'
df.to_csv(output_path, index=False)

print(f"✅ Data exported successfully to: {output_path}")
print(f"\n📊 Final Summary:")
print(f"  • Total records: {len(df):,}")
print(f"  • Time period: {df['datetime'].min()} to {df['datetime'].max()}")
print(f"  • Regions analyzed: {df['region'].nunique()}")
print(f"  • Surplus windows detected: {(df['surplus_mw'] > 0).sum():,} hours")
print(f"  • Total curtailed MWh potential: {df[df['surplus_mw'] > 0]['surplus_mw'].sum():,.0f}")